# Step 15 · Monitor in production

Every RAG / agent / API call is a trace: latency, failures, turn bad traces into tests.

*Same as the dashboard page, done in code.*

In [ ]:
# ── Setup (run first) ────────────────────────────────────────────────────────────
# Uses the same lab registry as the dashboard. To experiment in a sandbox instead:
#   os.environ["LAB_HOME"] = "/tmp/my_lab"   (before importing lab)
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT); os.chdir(ROOT)
os.environ.setdefault("HF_HUB_OFFLINE", "1")          # models are already cached locally
import pandas as pd
from lab import data, registry, paths, rewards
data.bootstrap_samples()
BASE = registry.ensure_base(paths.DEFAULT_BASE_MODEL)
if not registry.rewards():
    registry.add_reward(rewards.DEFAULT_RULES, note="Default draft rules")

def latest(kind):
    """Newest model id of a kind: base | sft | rl | rlhf | rm (or None)."""
    ms = [m for m in registry.models() if m["kind"] == kind]
    return ms[-1]["id"] if ms else None

def latest_ds(role):
    d = registry.latest_dataset(role)
    return d["id"] if d else None

print("Lab home:", paths.home(), "| base model:", BASE)

In [ ]:
tr = pd.DataFrame(registry.traces(limit=1000))
if tr.empty:
    print("No traces yet — run notebooks 12–14.")
else:
    display(tr.groupby("kind")["latency"].describe()[["count", "50%", "max"]])
    display(tr[["id", "kind", "source", "model_id", "input", "output", "latency"]].head(10))

In [ ]:
# Flag a bad trace and turn it into an agent test (uncomment to write)
# registry.update("traces", int(tr.id[0]), flag="bad")
# from lab import agent_evals
# agent_evals.add_scenario({"type": "tool", "turns": [tr.input[0]], "turn_tools": [["policy_search"]], "must_not_include": ["50%"]})